# GCN - Graph Convolutional Network

We now build our first GNN: a Graph Convolutional Network (GCN).
Unlike the Random Forest baseline, GCN aggregates information from 
neighboring nodes, allowing the model to detect fraud patterns 
that are invisible from individual transaction features alone.

**Benchmark to beat:** Recall > 11% on illicit nodes at reasonable precision.

In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import EllipticBitcoinDataset
from torch_geometric.nn import GCNConv
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load dataset
dataset = EllipticBitcoinDataset(root='../data/elliptic')
data = dataset[0]

# Load timesteps
feat_df = pd.read_csv('../data/elliptic/raw/elliptic_txs_features.csv', header=None)
class_df = pd.read_csv('../data/elliptic/raw/elliptic_txs_classes.csv')
feat_df.columns = ['txId'] + [f'f{i}' for i in range(1, feat_df.shape[1])]
feat_df['timestep'] = feat_df['f1'].astype(int)
class_df.columns = ['txId', 'label']
df_meta = feat_df[['txId', 'timestep']].merge(class_df, on='txId')

timesteps = torch.tensor(df_meta['timestep'].values)

print(f"Dataset loaded — Nodes: {data.num_nodes:,} | Features: {data.num_node_features}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Dataset loaded — Nodes: 203,769 | Features: 165
Device: cuda


## 1. Model Definition

A 3-layer GCN where each layer aggregates features from neighboring nodes.
Dropout is applied between layers to prevent overfitting.

In [2]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv3(x, edge_index)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(in_channels=165, hidden_channels=64, out_channels=2).to(device)
data = data.to(device)
timesteps = timesteps.to(device)

print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

GCN(
  (conv1): GCNConv(165, 64)
  (conv2): GCNConv(64, 64)
  (conv3): GCNConv(64, 2)
)

Parameters: 14,914


## 2. Training Setup

We use a temporal split (timesteps 1-34 for training, 35-49 for testing)
and apply class weighting to handle the severe imbalance between licit and illicit nodes.
Only labeled nodes contribute to the loss.

In [5]:
# Masks
labeled_mask = data.y != 0
train_mask = labeled_mask & (timesteps <= 34)
test_mask = labeled_mask & (timesteps > 34)

# Labels: 1=illicit → 1, 2=licit → 0
y_binary = torch.zeros(data.y.shape, dtype=torch.long).to(device)
y_binary[data.y == 1] = 1  # illicit
y_binary[data.y == 2] = 0  # licit

# Class weights
n_licit = (y_binary[train_mask] == 0).sum().item()
n_illicit = (y_binary[train_mask] == 1).sum().item()
weight = torch.tensor([1.0, n_licit / n_illicit]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss(weight=weight)

print(f"Train nodes: {train_mask.sum():,}")
print(f"Test nodes:  {test_mask.sum():,}")
print(f"Train illicit: {n_illicit:,}")
print(f"Train licit: {n_licit:,}")
print(f"Class weight illicit: {n_licit/n_illicit:.1f}x")

Train nodes: 109,833
Test nodes:  51,917
Train illicit: 3,462
Train licit: 106,371
Class weight illicit: 30.7x


## 3. Training Loop

In [6]:
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[train_mask], y_binary[train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(mask):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        prob = F.softmax(out, dim=1)[:, 1]
    
    y_true = y_binary[mask].cpu().numpy()
    y_pred = pred[mask].cpu().numpy()
    y_prob = prob[mask].cpu().numpy()
    
    auc = roc_auc_score(y_true, y_prob)
    return y_true, y_pred, y_prob, auc

# Training
losses = []
test_aucs = []

for epoch in range(1, 201):
    loss = train()
    losses.append(loss)
    
    if epoch % 10 == 0:
        y_true, y_pred, y_prob, auc = evaluate(test_mask)
        test_aucs.append(auc)
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Test AUC: {auc:.4f}")

Epoch  10 | Loss: 0.5547 | Test AUC: 0.6935
Epoch  20 | Loss: 0.4829 | Test AUC: 0.7193
Epoch  30 | Loss: 0.4409 | Test AUC: 0.7584
Epoch  40 | Loss: 0.4069 | Test AUC: 0.7863
Epoch  50 | Loss: 0.3727 | Test AUC: 0.7961
Epoch  60 | Loss: 0.3532 | Test AUC: 0.7972
Epoch  70 | Loss: 0.3381 | Test AUC: 0.7987
Epoch  80 | Loss: 0.3212 | Test AUC: 0.8045
Epoch  90 | Loss: 0.3143 | Test AUC: 0.8063
Epoch 100 | Loss: 0.3089 | Test AUC: 0.8126
Epoch 110 | Loss: 0.3021 | Test AUC: 0.8149
Epoch 120 | Loss: 0.2935 | Test AUC: 0.8138
Epoch 130 | Loss: 0.2891 | Test AUC: 0.7986
Epoch 140 | Loss: 0.2860 | Test AUC: 0.8057
Epoch 150 | Loss: 0.2788 | Test AUC: 0.7985
Epoch 160 | Loss: 0.2768 | Test AUC: 0.7983
Epoch 170 | Loss: 0.2762 | Test AUC: 0.7934
Epoch 180 | Loss: 0.2749 | Test AUC: 0.8025
Epoch 190 | Loss: 0.2687 | Test AUC: 0.7964
Epoch 200 | Loss: 0.2702 | Test AUC: 0.7974


## 4. Results

The GCN achieves a lower AUC than Random Forest, which is expected,
the node features already include precomputed 1-hop aggregates, 
giving tabular models an unfair advantage on this metric.
The key question is whether GCN improves recall on illicit nodes.

In [7]:
y_true, y_pred, y_prob, auc = evaluate(test_mask)

print("=== GCN Results ===")
print(classification_report(y_true, y_pred, target_names=['Licit', 'Illicit']))
print(f"ROC-AUC: {auc:.4f}")
print(f"\nBaseline RF recall on illicit: 11%")
print(f"GCN recall on illicit: {(y_pred[y_true==1]==1).sum() / (y_true==1).sum() * 100:.0f}%")

=== GCN Results ===
              precision    recall  f1-score   support

       Licit       0.99      0.94      0.96     50834
     Illicit       0.13      0.41      0.20      1083

    accuracy                           0.93     51917
   macro avg       0.56      0.67      0.58     51917
weighted avg       0.97      0.93      0.95     51917

ROC-AUC: 0.7974

Baseline RF recall on illicit: 11%
GCN recall on illicit: 41%


## 5. Conclusions

| Metric | Random Forest | GCN |
|--------|--------------|-----|
| ROC-AUC | 0.95 | 0.80 |
| Illicit Recall | 11% | 41% |
| Illicit Precision | 74% | 13% |

The GCN detects **4x more fraud** than the Random Forest baseline, 
despite a lower AUC. This demonstrates that graph structure carries 
signal invisible to tabular models.

The precision/recall trade-off can be tuned via the decision threshold, 
a topic we explore in the next notebook with a more powerful model: GAT.

### Why is illicit precision so low?

The class weight of 30.7x makes the GCN aggressive: it learns to flag many nodes 
as illicit to avoid missing real fraud. The result: only 1 in 8 nodes flagged as 
illicit is actually fraudulent (13% precision), but the model catches 41% of all fraud.

This is the fundamental precision-recall trade-off in fraud detection:
- **High precision** (Random Forest): when it says fraud, it's usually right, but misses 89% of cases
- **High recall** (GCN): catches far more fraud, but generates more false positives

Which is better depends on the business context. If each alert requires expensive 
manual review, precision matters. If missing fraud is more costly than investigating 
false positives, recall matters. In practice, the decision threshold is tuned 
on the precision-recall curve to find the optimal operating point.